# 프로토콜 v1.1 — E2c: bayesrul 공개 코드를 직접 실행 (탐색 분석, GPU)

Basora 외(RESS 2025)의 공개 코드 bayesrul(커밋 `2943f4d`)을 **원 설정 그대로** 실행해 HNN, MCD, DE의 시험 창 예측(평균과 표준편차)을 저장합니다. 채점과 PI-BDP와의 짝지은 검정은 다음 노트북(E2c bayesrul 채점, CPU)에서 합니다.

- **A단계 — 재현 확인 (v1.1 3.4절):** 원 논문 조건(DS01–DS05를 묶은 모델 하나)으로 HNN을 학습해 원 논문 표 7의 HNN 시험 RMSE(8.279 ± 0.181)와 비교합니다. 판정: 우리 평균이 8.279 ± max(2 × 0.181, 0.3) 안이면 "재현됨". 학습·검증·시험 창 수도 원 논문 표 3(238,182 / 26,464 / 144,015)과 비교합니다.
- **B단계 — 본 실행 (v1.1 3.3절):** 9개 서브셋을 묶어 HNN 시드 1–10, MCD 시드 1–5를 학습합니다. 학습률, dropout, 배치, 최대 500 epoch, patience 50 조기종료, dev의 10% 무작위 검증 분할은 원 설정(`ncmapss_hnn`, `ncmapss_mcd`, `scripts/train_*.sh`) 그대로입니다. 검증 NLL이 가장 좋은 체크포인트로 시험 창을 한 번 예측합니다(원 코드의 방식). 시험 데이터로 고르는 것은 없습니다.
- **DE:** 원 코드의 `deep_ensemble_gen`으로 HNN 10개 중 5개씩 묶은 앙상블 5개를 만듭니다(`random.seed(1)`). 추가 학습은 없습니다.
- **계산량 규칙 (실행 전에 정함, 시간만 사용):** 짧은 시간 측정 실행(1·3 epoch)으로 전체 GPU 시간을 추정합니다. 40시간을 넘으면 다음 순서로 줄입니다: ① A단계 HNN 5회 → 3회, ② MCD 5회 → 3회, ③ HNN 10회 → 6회(DE에 필요한 최소 수).

**원 코드는 고치지 않습니다.** 원 저장소의 학습 과제, 전처리, 모델, 예측 함수를 그대로 부르고, 이 노트북이 만드는 보조 스크립트 네 개(`br_*.py`)만 씁니다. 바뀐 점은 다음 셋뿐입니다.
1. 시드 지정: 원 `train.py`는 설정의 `seed` 값을 실제로 쓰지 않으므로, 학습 전에 `pl.seed_everything(seed)`를 부릅니다.
2. 호환 처리: 원 `train.py`가 불러오는 `LightningLoggerBase`(형 표시에만 쓰임)가 저장소에 고정된 pytorch-lightning 1.9.5에는 없어, 후속 클래스 이름으로 연결합니다.
3. 출력: 진행 막대를 끄고(화면 표시만 바뀜), 시험 창 예측을 창 순서대로 저장합니다. 각 창의 엔진·사이클은 원 전처리 함수로 다시 만들고, 시험 LMDB의 기록과 하나하나 맞는지 확인합니다.

**실행 전:** 이 노트북을 커밋한 주소를 `RUN_COMMIT_URL`에 넣으세요. **런타임:** GPU L4(CruiseBench와 같게), 고용량 RAM. 끊겨도 다시 처음부터 실행하면 끝난 실행은 건너뛰고, 학습 중이던 실행은 마지막 체크포인트에서 이어 갑니다.

In [ ]:
# === 1. 설정 ===
import os, sys, json, glob, time, hashlib, subprocess, zipfile, re, shutil
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'; WORK = '/content/ncmapss_raw'; LOCAL = '/content/e2c_br'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm'); WORK = os.environ.get('NCM_WORK', './ncmapss_raw'); LOCAL = os.environ.get('BR_LOCAL', './e2c_br')
OUT = f'{ROOT}/addendum_v11/e2c_bayesrul'
for d in [OUT, f'{OUT}/preds', f'{OUT}/runs', f'{OUT}/logs', WORK, LOCAL]: os.makedirs(d, exist_ok=True)
ADDENDUM_COMMIT = 'fb8db254edb1fa613a00c4c22b031c7c7515489d'      # 프로토콜 v1.1 커밋
RUN_COMMIT_URL = ''                                                # 이 노트북을 커밋한 주소
assert re.match(r'https://github\.com/.+/commit/[0-9a-f]{7,40}$', RUN_COMMIT_URL), '이 노트북을 먼저 커밋하고 주소를 넣으세요.'
SUBSETS = ['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008']
FILES = {'rep': [f'N-CMAPSS_{s}' for s in SUBSETS[:5]],          # 원 논문: DS01–DS05 (원 설정 파일과 같음)
         'main': [f'N-CMAPSS_{s}' for s in SUBSETS]}             # 본 실행: 9개 서브셋 (v1.1 3.1)
PAPER_HNN_RMSE = (8.279, 0.181)                                  # 원 논문 표 7 (평균 ± SD)
PAPER_COUNTS = dict(train=238182, val=26464, test=144015)        # 원 논문 표 3
PLAN_EPOCHS, BUDGET_H, CU_PER_H = 400, 40.0, 1.71                # 시간 추정용 epoch 수, GPU 시간 한도, L4 단위/시간(참고)
TEST_OVR = os.environ.get('BR_TEST_OVERRIDES', '').split()       # 로컬 시험용. Colab에서는 비어 있음
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() if shutil.which('nvidia-smi') else 'cpu'
print('GPU:', GPU or 'cpu')

In [ ]:
# === 2. N-CMAPSS 내려받기 (Phase 2b와 같은 파일인지 해시로 확인) ===
URL = 'https://phm-datasets.s3.amazonaws.com/NASA/17.+Turbofan+Engine+Degradation+Simulation+Data+Set+2.zip'
MANUAL_ZIP = ''   # 자동 내려받기가 안 되면 드라이브에 올린 zip 경로
def h5_list(): return sorted(glob.glob(os.path.join(WORK, '**', '*.h5'), recursive=True))
if not h5_list():
    zpath = os.path.join(WORK, 'ncmapss.zip')
    if MANUAL_ZIP: subprocess.run(['cp', MANUAL_ZIP, zpath])
    elif not (os.path.exists(zpath) and zipfile.is_zipfile(zpath)):
        subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', zpath, URL])
    done = set()
    while True:
        zs = [z for z in glob.glob(os.path.join(WORK, '**', '*.zip'), recursive=True) if z not in done]
        if not zs: break
        for z in zs:
            done.add(z)
            if zipfile.is_zipfile(z):
                with zipfile.ZipFile(z) as f: f.extractall(os.path.splitext(z)[0] + '_x')
    for z in glob.glob(os.path.join(WORK, '**', '*.zip'), recursive=True): os.remove(z)
H5 = {os.path.basename(p).replace('.h5', '').replace('N-CMAPSS_', ''): p for p in h5_list() if '/flat/' not in p}
REFH = json.load(open(f'{ROOT}/phase2b/sha256_h5.json'))
FLAT = os.path.join(WORK, 'flat'); os.makedirs(FLAT, exist_ok=True)
for s in SUBSETS:
    assert os.path.getsize(H5[s]) == REFH[s]['bytes'], f'{s}: 파일 크기가 Phase 2b와 다름'
    h = hashlib.sha256()
    with open(H5[s], 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''): h.update(b)
    assert h.hexdigest() == REFH[s]['sha256'], f'{s}: 해시가 Phase 2b와 다름'
    dst = os.path.join(FLAT, f'N-CMAPSS_{s}.h5')
    if not os.path.exists(dst): os.symlink(os.path.abspath(H5[s]), dst)
print('h5 9개 해시 확인 완료')

In [ ]:
# === 3. bayesrul 고정 커밋 받기 ===
BR = os.path.join(LOCAL, 'bayesrul'); BR_COMMIT = '2943f4d'
if not os.path.exists(BR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/lbasora/bayesrul', BR], check=True)
subprocess.run(['git', '-C', BR, 'checkout', '-q', BR_COMMIT], check=True)
BR_HEAD = subprocess.run(['git', '-C', BR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert BR_HEAD.startswith(BR_COMMIT)
def br_clean():
    st = subprocess.run(['git', '-C', BR, 'status', '--porcelain', '--untracked-files=no'], capture_output=True, text=True).stdout.strip()
    assert not st, f'원 코드가 바뀌어 있음: {st}'
br_clean(); print('bayesrul', BR_HEAD)

In [ ]:
# === 4. 실행 환경: 저장소 poetry.lock에 고정된 판 (Python 3.10, torch 1.12.0) ===
# 쓰는 모델(HNN·MCD)에 필요한 패키지만 설치합니다. 베이지안 신경망용 pyro·TyXe는 v1.1 2.2절에서 뺀 모델에만 쓰여 설치하지 않습니다.
LOCKED = ['pytorch-lightning==1.9.5', 'lightning-utilities==0.9.0', 'torchmetrics==1.1.2', 'hydra-core==1.3.2',
          'hydra-colorlog==1.2.0', 'omegaconf==2.3.0', 'optuna==2.10.1', 'lmdb==1.4.1', 'numpy==1.25.2', 'pandas==1.5.3',
          'scikit-learn==1.3.0', 'scipy==1.11.2', 'h5py==3.9.0', 'tqdm==4.66.1', 'pyrootutils==1.0.4', 'python-dotenv==1.0.0',
          'rich==12.6.0', 'tensorboard==2.14.0', 'shapely==2.0.1', 'uncertainty-toolbox==0.1.1', 'statsmodels==0.13.5',
          'pyarrow==8.0.0', 'torchinfo==1.8.0', 'colorlog==6.7.0', 'matplotlib==3.7.3', 'protobuf==4.24.3', 'setuptools<70']
PY = os.environ.get('BR_VENV_PY') or os.path.join(LOCAL, 'venv', 'bin', 'python')
if not os.path.exists(PY):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
    subprocess.run(['uv', 'python', 'install', '3.10'], check=True)
    subprocess.run(['uv', 'venv', '-q', '--python', '3.10', os.path.join(LOCAL, 'venv')], check=True)
    subprocess.run(['uv', 'pip', 'install', '-q', '--python', PY, 'torch==1.12.0+cu116',
                    '--index-url', 'https://download.pytorch.org/whl/cu116'], check=True)
    subprocess.run(['uv', 'pip', 'install', '-q', '--python', PY] + LOCKED, check=True)
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join([BR] + os.environ.get('BR_EXTRA_PYTHONPATH', '').split(os.pathsep)).strip(os.pathsep),
           HYDRA_FULL_ERROR='1', PYTHONUNBUFFERED='1')
chk = subprocess.run([PY, '-c', 'import torch, pytorch_lightning as pl, hydra, bayesrul; '
                      'print(torch.__version__, pl.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")'],
                     capture_output=True, text=True, env=ENV, cwd=BR)
print(chk.stdout, chk.stderr[-2000:]); assert chk.returncode == 0
if IN_COLAB: assert 'True' in chk.stdout, 'GPU가 보이지 않습니다. 런타임을 GPU로 바꾸세요.'
open(f'{OUT}/env_freeze.txt', 'w').write(subprocess.run(['uv', 'pip', 'freeze', '--python', PY], capture_output=True, text=True).stdout
                                         if shutil.which('uv') else subprocess.run([PY, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)

In [ ]:
# === 5. 보조 스크립트 쓰기 (원 저장소 밖에 둡니다) ===
TOOL_SRC = {'br_train.py': "# E2c bayesrul: run the original training task with the seed applied.\n# Changes: pl.seed_everything(cfg.seed) (the original train.py never uses cfg.seed) and the import shim below.\nimport os\nimport hydra\nimport pytorch_lightning as pl\nimport pytorch_lightning.loggers as pll\nfrom omegaconf import DictConfig\n# Compatibility shim: train.py imports LightningLoggerBase (used only as a type hint), which the locked\n# pytorch-lightning 1.9.5 no longer has. Alias it to its successor class; nothing else changes.\nif not hasattr(pll, 'LightningLoggerBase'):\n    pll.LightningLoggerBase = pll.Logger\nfrom bayesrul.tasks import train as T          # importing sets PROJECT_ROOT (pyrootutils), as in the original\n\nCONF = os.path.abspath(os.path.join(os.path.dirname(T.__file__), '..', 'conf'))\n\n\n@hydra.main(version_base=None, config_path=CONF, config_name='train')\ndef main(cfg: DictConfig) -> None:\n    pl.seed_everything(cfg.seed, workers=True)\n    T.train(cfg, trial=None)\n\n\nif __name__ == '__main__':\n    main()\n", 'br_predict.py': '# E2c bayesrul: test-window predictions from one training run.\n# The model is rebuilt from that run\'s own saved hydra config, the best-validation checkpoint\n# (the original predict task\'s "epoch*.ckpt") is loaded, and the original HNN.predict_step is used.\n# Output keeps the test LMDB order (the original predict task does the same, shuffle=False).\nimport argparse, glob, os\nimport hydra\nimport pandas as pd\nimport pytorch_lightning as pl\nimport torch\nfrom omegaconf import OmegaConf\nfrom bayesrul.data.ncmapss.dataset import NCMAPSSDataModule\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--run_dir\', required=True)\nap.add_argument(\'--data_path\', required=True)\nap.add_argument(\'--out\', required=True)\nap.add_argument(\'--seed\', type=int, required=True)\nap.add_argument(\'--num_workers\', type=int, default=12)\na = ap.parse_args()\n\ncfg = OmegaConf.load(os.path.join(a.run_dir, \'.hydra\', \'config.yaml\'))\nck = sorted(glob.glob(os.path.join(a.run_dir, \'checkpoints\', \'epoch*.ckpt\')))\nassert len(ck) == 1, f\'expected one best checkpoint, found {ck}\'\ndm = NCMAPSSDataModule(a.data_path, batch_size=cfg.datamodule.batch_size, test_batch_size=10000,\n                       all_dset=False, num_workers=a.num_workers, pin_memory=torch.cuda.is_available())\ndm.set_predict_dataset(\'test\')\nmcfg = cfg.model\nmcfg.net.win_length = dm.win_length\nmcfg.net.n_features = dm.n_features\nmodel = hydra.utils.instantiate(mcfg, _convert_=\'partial\')\npl.seed_everything(a.seed, workers=True)          # MC dropout sampling at prediction is reproducible\ntrainer = pl.Trainer(accelerator=\'gpu\' if torch.cuda.is_available() else \'cpu\', devices=1, logger=False,\n                     enable_progress_bar=False, enable_model_summary=False)\npreds = trainer.predict(model=model, datamodule=dm, ckpt_path=ck[0])\ndf = pd.DataFrame.from_records(preds)\ndf = df.explode(df.columns.tolist()).reset_index(drop=True).astype(float)\ndf.insert(0, \'idx\', range(len(df)))\ndf.to_csv(a.out, index=False)\nprint(f\'{len(df)} windows, checkpoint {os.path.basename(ck[0])}, mc_samples={mcfg.mc_samples}, dropout={mcfg.net.dropout}\')\n', 'br_windows.py': "# E2c bayesrul: rebuild the (file, unit, cycle) of every test window with the original\n# preprocessing functions, then check it against the test LMDB record by record.\n# Window k of a unit ends at row sl.stop - 1 of the 0.1 Hz unit trajectory; its label is the\n# piece-wise RUL of that row (original process_dataframe).\nimport argparse, json\nfrom pathlib import Path\nimport lmdb\nimport numpy as np\nimport pandas as pd\nfrom bayesrul.data.ncmapss.preprocessing import read_hdf5, linear_piece_wise_RUL\nfrom bayesrul.data.lmdb_utils import make_slice\n\nap = argparse.ArgumentParser()\nap.add_argument('--hdf5_path', required=True)\nap.add_argument('--data_path', required=True)        # folder holding lmdb/\nap.add_argument('--out', required=True)\nap.add_argument('--counts_out', required=True)\nap.add_argument('--files', nargs='+', required=True)\nap.add_argument('--win', type=int, default=30)\nap.add_argument('--step', type=int, default=10)\nap.add_argument('--skip', type=int, default=10)\na = ap.parse_args()\n\nrows = []; extra = {}\nfor fname in a.files:\n    raw = read_hdf5(Path(a.hdf5_path, fname), 'test', np.float32, a.files, vars=['W', 'X_s', 'A'])\n    yc = raw.groupby(['unit', 'cycle']).rul.agg(['min', 'max'])\n    assert (yc['min'] == yc['max']).all(), 'raw RUL not constant within a cycle'\n    df = linear_piece_wise_RUL(raw.copy())\n    # The original merge duplicates rows that are identical in every column (k copies -> k*k rows).\n    # The LMDB was built from this frame, so the windows are rebuilt from it as is; the count is recorded.\n    extra[fname] = int(len(df) - len(raw))\n    df['rul_raw'] = yc['min'].reindex(pd.MultiIndex.from_arrays([df['unit'], df['cycle']])).values\n    df = df[::a.skip]\n    ds_id = int(str(Path(fname).stem).split('_')[-1].split('-')[0][3])     # same expression as process_files\n    for unit, traj in df.groupby('unit'):\n        cyc = traj['cycle'].values; lab = traj['rul'].values; rr = traj['rul_raw'].values\n        for k, sl in enumerate(make_slice(traj.shape[0], a.win, a.step)):\n            e = sl.stop - 1\n            rows.append((fname, ds_id, int(unit), k, int(cyc[sl.start]), int(cyc[e]), float(lab[e]), float(rr[e])))\nW = pd.DataFrame(rows, columns=['file', 'ds_id', 'unit', 'win_id', 'cycle_start', 'cycle', 'label', 'rul_raw'])\nW.insert(0, 'idx', range(len(W)))\n\nenv = lmdb.open(str(Path(a.data_path, 'lmdb', 'test.lmdb')), readonly=True, lock=False)\nwith env.begin() as t:\n    n = int(t.get(b'nb_lines').decode())\n    assert n == len(W), f'LMDB has {n} test windows, rebuilt {len(W)}'\n    g = lambda k, i: t.get(f'{k}_{i}'.encode()).decode()\n    ds = np.array([int(g('ds_id', i)) for i in range(n)])\n    un = np.array([int(float(g('unit_id', i))) for i in range(n)])\n    wi = np.array([int(g('win_id', i)) for i in range(n)])\n    ru = np.array([float(g('rul', i)) for i in range(n)])\nassert (ds == W.ds_id.values).all() and (un == W.unit.values).all() and (wi == W.win_id.values).all(), 'ids differ'\nassert np.allclose(ru, W.label.values), 'labels differ'\nW.to_csv(a.out, index=False)\ncounts = {}\nfor name in ['train', 'val', 'test']:\n    p = Path(a.data_path, 'lmdb', f'{name}.lmdb')\n    if p.exists():\n        e = lmdb.open(str(p), readonly=True, lock=False)\n        with e.begin() as t: counts[name] = int(t.get(b'nb_lines').decode())\ncounts['extra_rows_from_piecewise_merge'] = extra\njson.dump(counts, open(a.counts_out, 'w'), indent=1)\nprint(f'{len(W)} test windows rebuilt and matched to the LMDB; counts {counts}')\n", 'br_de.py': "# E2c bayesrul: deep ensembles from the HNN runs with the original deep_ensemble_gen\n# (5 HNN per ensemble, 5 ensembles, random.seed(1)), and the original metrics (compute_metrics).\nimport argparse, glob, json, os, random\nfrom itertools import combinations\nimport numpy as np\nimport pandas as pd\nfrom bayesrul.models.deepens import deep_ensemble_gen\n\nap = argparse.ArgumentParser()\nap.add_argument('--pred_dir', required=True)\nap.add_argument('--tag', required=True)              # main / rep\nap.add_argument('--out_metrics', required=True)\na = ap.parse_args()\n\ndef load(method):\n    fs = sorted(glob.glob(os.path.join(a.pred_dir, f'{a.tag}_{method}_[0-9][0-9][0-9].csv.gz')))\n    return {os.path.basename(f).split('.')[0].split('_', 1)[1]: pd.read_csv(f) for f in fs}\n\nH = load('HNN'); M = load('MCD')\nn = len(H); assert list(H) == [f'HNN_{i:03d}' for i in range(n)], list(H)\ninfo = {}\nif n >= 5 and a.tag == 'main':\n    df = pd.concat([d.assign(model=k, method='HNN') for k, d in H.items()], ignore_index=True)\n    random.seed(1)\n    members = [list(c) for c in random.sample(list(combinations(range(n), 5)), 5)]   # same draw as deep_ensemble_gen\n    for i, de in enumerate(deep_ensemble_gen(df, ['HNN'], 5, 5)):\n        mu = np.mean([H[f'HNN_{j:03d}'].preds.values for j in members[i]], axis=0)\n        assert np.allclose(mu, de.preds.values), 'DE member check failed'\n        de = de[['preds', 'labels', 'stds']].copy(); de.insert(0, 'idx', range(len(de)))\n        de.to_csv(os.path.join(a.pred_dir, f'{a.tag}_DE_{i:03d}.csv.gz'), index=False)\n    info['de_members'] = {f'DE_{i:03d}': [f'HNN_{j:03d}' for j in m] for i, m in enumerate(members)}\n    json.dump(info, open(os.path.join(a.pred_dir, f'{a.tag}_de_members.json'), 'w'), indent=1)\nD = load('DE')\n\nrows = []\ntry:\n    import torch\n    from bayesrul.results.metrics import compute_metrics          # original metrics (needs cuda:0)\n    for meth, dd in [('HNN', H), ('MCD', M), ('DE', D)]:\n        for k, d in dd.items():\n            m = compute_metrics(d)\n            rows.append(dict(method=meth, model=k, **{x: m[x] for x in ['mae', 'rmse', 'nll', 'rmsce', 'ece', 'sharp', 's']}))\n    pd.DataFrame(rows).to_csv(a.out_metrics, index=False)\n    print(pd.DataFrame(rows).groupby('method')[['rmse', 'nll', 'rmsce', 'sharp']].agg(['mean', 'std']).round(3).to_string())\nexcept Exception as ex:\n    print('original metrics not computed:', repr(ex))\nprint('HNN', n, '| MCD', len(M), '| DE', len(D), info.get('de_members', ''))\n"}
TOOLS = os.path.join(LOCAL, 'tools'); os.makedirs(TOOLS, exist_ok=True)
for n, src in TOOL_SRC.items(): open(os.path.join(TOOLS, n), 'w').write(src)
json.dump({n: hashlib.sha256(s.encode()).hexdigest() for n, s in TOOL_SRC.items()}, open(f'{OUT}/tools_sha256.json', 'w'), indent=1)

def sh(args, log, tail=40):
    # 원 코드를 3.10 환경에서 실행. 출력은 로그 파일로, 실패하면 끝부분을 보여 주고 멈춤.
    with open(log, 'a') as f:
        f.write(f'\n### {time.strftime("%Y-%m-%d %H:%M:%S")} {" ".join(args)}\n'); f.flush()
        r = subprocess.run(args, stdout=f, stderr=subprocess.STDOUT, env=ENV, cwd=BR)
    if r.returncode != 0:
        print(''.join(open(log).readlines()[-tail:])); raise RuntimeError(f'실패: {" ".join(args[:4])} ... (로그 {log})')
print('보조 스크립트', list(TOOL_SRC))

In [ ]:
# === 6. 데이터셋 만들기 (원 build_ds 과제) + 시험 창의 엔진·사이클 복원 ===
# 원 전처리는 정규화 평균·분산을 섞인 순서로 더하므로 다시 만들면 마지막 자릿수가 달라질 수 있습니다.
# 그래서 처음 만든 LMDB를 드라이브에 묶어 두고(약 2–3 GB), 세션이 바뀌면 그것을 풀어 씁니다. 내용 지문으로 같은지 확인합니다.
STORE = f'{ROOT}/addendum_v11/e2c_bayesrul_lmdb'; os.makedirs(STORE, exist_ok=True)
def lmdb_fingerprint(path):
    code = ('import lmdb, hashlib, sys\n'
            'h = hashlib.sha256()\n'
            'for n in ["train", "val", "test"]:\n'
            '    e = lmdb.open(f"{sys.argv[1]}/lmdb/{n}.lmdb", readonly=True, lock=False)\n'
            '    with e.begin() as t:\n'
            '        for k, v in t.cursor(): h.update(k); h.update(v)\n'
            'print(h.hexdigest())')
    return subprocess.run([PY, '-c', code, path], capture_output=True, text=True, env=ENV, cwd=BR, check=True).stdout.strip()

DATA = {}
for tag, files in FILES.items():
    d = os.path.join(LOCAL, f'data_{tag}'); DATA[tag] = d; tar = f'{STORE}/{tag}.tar'; ff = f'{OUT}/lmdb_fingerprint_{tag}.json'
    if not os.path.exists(f'{d}/BUILT'):
        shutil.rmtree(d, ignore_errors=True); os.makedirs(d); t = time.time()
        if os.path.exists(tar) and os.path.exists(ff):                      # 드라이브에 둔 것을 풀어 씀
            subprocess.run(['tar', '-xf', tar, '-C', d], check=True)
            assert lmdb_fingerprint(d) == json.load(open(ff))['sha256'], f'{tag}: 드라이브의 LMDB 지문이 다름'
        else:                                                               # 처음 한 번만 만듦
            sh([PY, '-m', 'bayesrul.tasks.build_ds', f'dataset.files=[{",".join(files)}]', f'dataset.hdf5_path={FLAT}',
                f'paths.output_dir={d}', f'hydra.run.dir={d}/_hydra'], f'{OUT}/logs/build_ds_{tag}.log')
            json.dump(dict(sha256=lmdb_fingerprint(d), files=files), open(ff, 'w'), indent=1)
            subprocess.run(['tar', '-cf', tar + '.part', '-C', d, 'lmdb', 'fc.csv'], check=True); os.replace(tar + '.part', tar)
        open(f'{d}/BUILT', 'w').write(json.load(open(ff))['sha256']); print(f'{tag}: 데이터셋 준비 {(time.time() - t) / 60:.1f}분')
    wf = f'{OUT}/windows_{tag}.csv.gz'
    if not os.path.exists(wf):
        sh([PY, f'{TOOLS}/br_windows.py', '--hdf5_path', FLAT, '--data_path', d, '--out', wf,
            '--counts_out', f'{OUT}/counts_{tag}.json', '--files'] + files, f'{OUT}/logs/windows_{tag}.log')
    print(tag, json.load(open(f'{OUT}/counts_{tag}.json')))
CNT = json.load(open(f'{OUT}/counts_rep.json'))
print('원 논문 표 3과 창 수 비교 (DS01–DS05):', {k: (CNT[k], PAPER_COUNTS[k], CNT[k] == PAPER_COUNTS[k]) for k in PAPER_COUNTS})
br_clean()

In [ ]:
# === 7. 학습·예측 함수 ===
EXP = {'HNN': ['experiment=ncmapss_hnn', 'trainer.deterministic=False', 'task_name=train_hnn'],                       # scripts/train_hnn.sh
       'MCD': ['experiment=ncmapss_mcd', 'model.mc_samples=20', 'trainer.deterministic=False', 'task_name=train_mcd']}  # scripts/train_mcd.sh
DISPLAY = ['~callbacks.rich_progress_bar', '+trainer.enable_progress_bar=false']   # 화면 표시만 끔
RUNLOG = f'{OUT}/run_log.csv'

def train_cmd(tag, method, seed, run_dir, extra=()):
    keys = {e.split('=')[0] for e in extra}
    return [PY, f'{TOOLS}/br_train.py'] + EXP[method] + [f'seed={seed}', f'datamodule.data_path={DATA[tag]}',
            f'hydra.run.dir={run_dir}'] + DISPLAY + list(extra) + [o for o in TEST_OVR if o.split('=')[0] not in keys]

def run_one(tag, method, seed):
    run = f'{seed - 1:03d}'; rd = f'{OUT}/runs/{tag}/{method}/{run}'; pf = f'{OUT}/preds/{tag}_{method}_{run}.csv.gz'
    if os.path.exists(pf): return
    resumed = False; t = time.time()
    if not os.path.exists(f'{rd}/DONE'):
        last = f'{rd}/checkpoints/last.ckpt'
        if os.path.exists(last): resumed = True; extra = [f'ckpt_path={last}']           # 끊긴 실행은 이어서
        else: shutil.rmtree(rd, ignore_errors=True); extra = []
        sh(train_cmd(tag, method, seed, rd, extra), f'{OUT}/logs/{tag}_{method}_{run}.log')
        open(f'{rd}/DONE', 'w').write(time.strftime('%Y-%m-%d %H:%M:%S'))
    sh([PY, f'{TOOLS}/br_predict.py', '--run_dir', rd, '--data_path', DATA[tag], '--out', pf, '--seed', str(seed)],
       f'{OUT}/logs/{tag}_{method}_{run}.log')
    best = os.path.basename(glob.glob(f'{rd}/checkpoints/epoch*.ckpt')[0])
    pd.DataFrame([dict(tag=tag, method=method, seed=seed, run=run, minutes=(time.time() - t) / 60, resumed=resumed, best_ckpt=best,
                       gpu=GPU, finished=time.strftime('%Y-%m-%d %H:%M:%S'))]).to_csv(RUNLOG, mode='a', header=not os.path.exists(RUNLOG), index=False)
    print(tag, method, seed, f'{(time.time() - t) / 60:.1f}분', best, '(이어서 학습)' if resumed else '')

In [ ]:
# === 8. 계산량 규칙: 시간 측정 실행 (성능은 보지 않음) ===
DEC = f'{OUT}/br_compute_decision.json'
if not os.path.exists(DEC):
    tm = {}
    for method in ['HNN', 'MCD']:
        T = {}
        for ep in [1, 3]:
            rd = os.path.join(LOCAL, 'probe', f'{method}_{ep}'); shutil.rmtree(rd, ignore_errors=True); t = time.time()
            sh(train_cmd('main', method, 1, rd, [f'trainer.max_epochs={ep}', 'trainer.min_epochs=1', 'test=False']), f'{OUT}/logs/probe.log')
            T[ep] = time.time() - t
        tm[method] = max((T[3] - T[1]) / 2, 1e-3)
    C9 = json.load(open(f'{OUT}/counts_main.json')); C5 = json.load(open(f'{OUT}/counts_rep.json'))
    ratio = C5['train'] / C9['train']
    def est(r, nm, nh): return (r * tm['HNN'] * ratio + nh * tm['HNN'] + nm * tm['MCD']) * PLAN_EPOCHS / 3600
    plan = [(5, 5, 10), (3, 5, 10), (3, 3, 10), (3, 3, 6)]        # (A단계 HNN, MCD, HNN) 줄이는 순서
    pick = next((p for p in plan if est(*p) <= BUDGET_H), plan[-1])
    dec = dict(sec_per_epoch=tm, plan_epochs=PLAN_EPOCHS, budget_gpu_h=BUDGET_H, gpu=GPU,
               estimates_gpu_h={str(p): round(est(*p), 1) for p in plan}, n_rep_hnn=pick[0], n_mcd=pick[1], n_hnn=pick[2],
               over_budget_even_after_reduction=bool(est(*pick) > BUDGET_H),
               rule='estimated GPU-h > 40 -> reduce: replication HNN 5->3, MCD 5->3, HNN 10->6 (timing only)')
    json.dump(dec, open(DEC, 'w'), indent=1)
dec = json.load(open(DEC)); print(json.dumps(dec, indent=1))
p = (dec['n_rep_hnn'], dec['n_mcd'], dec['n_hnn'])
print(f"예상 {dec['estimates_gpu_h'][str(p)]} GPU시간 ≈ {dec['estimates_gpu_h'][str(p)] * CU_PER_H:.0f} 컴퓨팅 단위(L4 기준, 참고)")

In [ ]:
# === 9. A단계: 재현 확인 (DS01–DS05, HNN) ===
GO = False   # 8번 셀의 예상 시간과 남은 컴퓨팅 단위를 확인한 뒤 True로 바꾸고 이 셀부터 실행하세요 (실행 수는 8번 규칙이 정함)
assert GO or os.environ.get('BR_GO'), '8번 셀의 예상 시간을 확인한 뒤 GO = True로 바꾸세요.'
for sd in range(1, dec['n_rep_hnn'] + 1): run_one('rep', 'HNN', sd)
W5 = pd.read_csv(f'{OUT}/windows_rep.csv.gz')
rm = []
for sd in range(1, dec['n_rep_hnn'] + 1):
    P = pd.read_csv(f'{OUT}/preds/rep_HNN_{sd - 1:03d}.csv.gz'); assert np.allclose(P.labels, W5.label)
    rm.append(float(np.sqrt(np.mean((P.preds - P.labels) ** 2))))        # 원 논문처럼 전체 시험 창, 상한 레이블
mu, sdv = PAPER_HNN_RMSE; tol = max(2 * sdv, 0.3); ours = float(np.mean(rm))
REP = pd.DataFrame([dict(model='HNN (DS01-DS05)', paper_mean=mu, paper_sd=sdv, ours_mean=ours, ours_sd=float(np.std(rm, ddof=1)) if len(rm) > 1 else np.nan,
                         n_runs=len(rm), runs=' '.join(f'{x:.3f}' for x in rm), tolerance=tol, reproduced=bool(abs(ours - mu) <= tol),
                         counts_match_table3=all(CNT[k] == PAPER_COUNTS[k] for k in PAPER_COUNTS))])
REP.to_csv(f'{OUT}/e2c_br_replication.csv', index=False); print(REP.T.to_string())
print('재현 판정:', '재현됨' if REP.reproduced[0] else '재현되지 않음 → B단계 전에 결과를 알려 주세요 (v1.1 3.4절: 원인 확인 후 한 번 더)')

In [ ]:
# === 10. B단계: 본 실행 (9개 서브셋 묶음) ===
for sd in range(1, dec['n_hnn'] + 1): run_one('main', 'HNN', sd)
for sd in range(1, dec['n_mcd'] + 1): run_one('main', 'MCD', sd)
print('B단계 완료:', sorted(os.path.basename(p) for p in glob.glob(f'{OUT}/preds/main_*.csv.gz')))

In [ ]:
# === 11. DE 만들기와 원 논문 지표 (원 코드 deep_ensemble_gen, compute_metrics) ===
for tag in ['main', 'rep']:
    sh([PY, f'{TOOLS}/br_de.py', '--pred_dir', f'{OUT}/preds', '--tag', tag, '--out_metrics', f'{OUT}/e2c_br_original_metrics_{tag}.csv'],
       f'{OUT}/logs/de_metrics_{tag}.log')
    print(''.join(open(f'{OUT}/logs/de_metrics_{tag}.log').readlines()[-12:]))
br_clean()

In [ ]:
# === 12. 기록과 해시 (채점은 다음 노트북) ===
files = [p for p in sorted(glob.glob(f'{OUT}/**/*', recursive=True)) if os.path.isfile(p) and not p.endswith('sha256_e2c_bayesrul.json')]
H = {}
for p in files:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    H[os.path.relpath(p, OUT)] = h.hexdigest()
json.dump(dict(addendum=ADDENDUM_COMMIT, run_commit=RUN_COMMIT_URL, bayesrul=BR_HEAD, gpu=GPU, files=H),
          open(f'{OUT}/sha256_e2c_bayesrul.json', 'w'), indent=1)
print('파일', len(H), '개 해시 기록')